In [ ]:
# Practice Problems Day 4

### Author: Joanna Bieri (solutions)

## Introduction to Machine Learning

#### University of Redlands - DATA 301
#### Prof: Joanna Bieri [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
#### Day 4 Practice - SOLUTIONS
#### [Class Website](https://joannabieri.com/machine_learning.html)

---

**These are my solutions.** Try every problem yourself first, then come here to check your work. There is more than one right way to write the code, and your words will be different from mine, that is fine! If your numbers do not match, look for the difference before you look at my code.

**Reading:** Geron, chapter 2, the section on **Create a Test Set**, and chapter 3, the section on **Measuring Accuracy Using Cross-Validation**

GOALS:

1. Get comfortable with `cross_val_score`, `GridSearchCV` and pipelines.
2. Learn to spot leakage, which is the point of the whole day.
3. See why stratification matters when the thing you care about is rare.

---

## Problem 1: Cross validation by hand

Use the same synthetic data from the Day 4 notes (the parabola with noise).

**1a.** Build a pipeline with `PolynomialFeatures(degree=8)`, `StandardScaler` and
`Ridge(alpha=1)`. Run `cross_val_score` with `cv=5` and print all five fold scores.

**1b.** Now run it again with `cv=10`. Print the ten scores.

**1c.** Compare the mean and the standard deviation for `cv=5` and `cv=10`. Which
gives the larger mean? Which gives the larger spread? Explain why the spread changes
in the direction it does.

**1d.** How many times was the model fit in total across both runs?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge

rng = np.random.default_rng(seed=42)
m = 200
X = 6 * rng.random((m, 1)) - 3
y = (0.5 * X**2 + X + 2 + rng.standard_normal((m, 1))).ravel()

model = make_pipeline(PolynomialFeatures(degree=8, include_bias=False),
                      StandardScaler(), Ridge(alpha=1.0))

for k in (5, 10):
    s = -cross_val_score(model, X, y, cv=k, scoring="neg_root_mean_squared_error")
    print(f"cv={k:2d}  mean {s.mean():.4f}   std {s.std():.4f}")
    print(f"       folds {[round(float(v), 3) for v in s]}\n")

**1a and 1b.** The fold scores are printed above, as RMSE (smaller is better).

**1c.** The means are almost the same, 0.9950 for `cv=5` and 0.9946 for `cv=10`. That makes sense, they are both estimating the same thing.

`cv=10` has the **larger spread**, 0.1726 against 0.0773. People often get the reason backwards! With 10 folds each test fold is only 20 points instead of 40, so each fold score is measured on less data and bounces around more.

The mean of the ten is still a slightly better estimate, because each model trained on 180 points instead of 160 and there are more scores to average. More folds gives you a better mean **and** noisier individual folds, at the same time.

**1d.** 5 + 10 = **15** fits.

---

## Problem 2: Find the leak

Each of the five snippets below has a leak, or does not. For each one, say
**leak or no leak**, and if it leaks, say exactly what information got where it
should not have.

No code needed. Write sentences.

**2a.**
```python
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y)
```

**2b.**
```python
X_train, X_test, y_train, y_test = train_test_split(X, y)
model = make_pipeline(StandardScaler(), Ridge())
scores = cross_val_score(model, X_train, y_train, cv=5)
```

**2c.** You are predicting which students will drop a course. One feature is the
number of assignments they submitted during the whole semester.

**2d.**
```python
# 200 songs by 20 artists, predicting whether a song is a hit
X_train, X_test, y_train, y_test = train_test_split(songs, hits, test_size=0.25)
```

**2e.**
```python
X_train, X_test, y_train, y_test = train_test_split(X, y)
best_k = 0
for k in [5, 10, 20, 50]:
    sel = SelectKBest(k=k).fit(X_train, y_train)
    score = model.fit(sel.transform(X_train), y_train).score(sel.transform(X_test), y_test)
    if score > best_k: best_k = k     # pick k using the test set
```


---

**2a. Leak.** The scaler computed its mean and standard deviation from all the data, including the rows that later became the test set.

**2b. No leak.** The split happened first, and `cross_val_score` refits the whole pipeline inside each fold, so the scaler only ever sees that fold's training data. This is the right way to do it.

**2c. Leak, and it is a time leak.** The number of assignments submitted over the whole semester includes the weeks after a student dropped. A student who drops in week 3 submits nothing for the rest of the term, so the feature already contains part of the answer. When you actually need the prediction, in week 3, that number does not exist yet.

**2d. Leak.** 200 songs but only 20 artists, so a random split by song puts the same artist in both the training and the test set. The model can learn "this artist makes hits" and look good without knowing anything about songs. Split by **artist** instead, with `GroupKFold`. This is the same idea as the patient scans in the notes.

**2e. Leak.** `k` is chosen by whichever value scored best on the test set. The test set has been used to make a choice, so its score is no longer honest. Choose `k` with a validation set or with cross validation on the training data.

---

## Problem 3: Build the leak yourself

Reproduce the noise experiment from the notes, but smaller, and see where the leak
disappears.

**3a.** Make a dataset of 100 samples and **200** features of pure noise, with coin
flip labels. Confirm there is no signal.

**3b.** Select the best 10 features using all the data, then cross validate. What
accuracy do you get?

**3c.** Do it properly with the selection inside a pipeline. What accuracy now?

**3d.** The notes used 5000 features and got 0.850 the wrong way. You used 200.
Is your fake accuracy bigger or smaller than the notes got? Explain the pattern in
one sentence: what makes the leak worse?

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression

r = np.random.default_rng(seed=0)
X_noise = r.standard_normal((100, 200))     # only 200 features this time
y_coin = r.integers(0, 2, 100)

sel = SelectKBest(f_classif, k=10).fit(X_noise, y_coin)
wrong = cross_val_score(LogisticRegression(max_iter=5000),
                        sel.transform(X_noise), y_coin, cv=5).mean()

right = cross_val_score(make_pipeline(SelectKBest(f_classif, k=10),
                                      LogisticRegression(max_iter=5000)),
                        X_noise, y_coin, cv=5).mean()

print(f"200 features, k=10")
print(f"  wrong (select first) : {wrong:.3f}")
print(f"  right (in pipeline)  : {right:.3f}")
print(f"  invented             : {wrong-right:+.3f}")

print("\nhow the leak grows with the number of features to search:")
for p in (50, 200, 1000, 5000):
    Xp = r.standard_normal((100, p)); yp = r.integers(0, 2, 100)
    s = SelectKBest(f_classif, k=10).fit(Xp, yp)
    w = cross_val_score(LogisticRegression(max_iter=5000), s.transform(Xp), yp, cv=5).mean()
    print(f"  {p:5d} features: leaky accuracy {w:.3f}")

**3a.** The features and labels are random numbers, so there is nothing to learn. The honest accuracy should be about 0.5, a coin flip.

**3b.** Selecting the features first, using all the data, gives **0.700**. That is a made up 70 percent on pure noise!

**3c.** With the selection inside the pipeline it drops to **0.540**, about a coin flip, which is the truth.

**3d.** My fake accuracy is **smaller** with 200 features (0.700) than the 0.850 the notes got with 5000.

BEWARE - do not read too much into the sweep at the bottom. It is not a clean staircase: 50 features gives 0.710 and 200 gives 0.700, so at the small end it is just noise. The trend only shows up across the big jumps, 200 to 1000 to 5000 giving 0.700, 0.750, 0.810.

The pattern that is real: **the more features you search through, the more noise you can find that happens to line up with the labels.** With 5000 columns and 100 rows, some column will match the labels by pure luck. So this is most dangerous for **wide** data, with many features and few rows, eg. genetics data, sensor data, or text with one column per word.

---

## Problem 4: Stratification

**4a.** Make a classification dataset with 300 samples where only **6 percent** are
positive. Give the positive class a real signal so a model can learn something.

**4b.** Split it with `KFold(n_splits=5, shuffle=True)` and print how many positives
land in each test fold. Try at least three different `random_state` values.

**4c.** Do the same with `StratifiedKFold`. What changes?

**4d.** Did any of your `KFold` runs produce a fold with **zero** positives? If yes,
say what score a model that answers "negative" for every single case would get on that
fold, and why that is a problem. If no, explain how you could make it more likely.

In [ ]:
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.datasets import make_classification

X_rare, y_rare = make_classification(
    n_samples=300, n_features=6, n_informative=3, n_redundant=0,
    weights=[0.94, 0.06], flip_y=0, random_state=1)
print(f"{y_rare.sum()} positives out of {len(y_rare)} ({y_rare.mean():.1%})\n")

for state in (0, 1, 2, 3):
    kf = [int(y_rare[te].sum()) for _, te in
          KFold(5, shuffle=True, random_state=state).split(X_rare, y_rare)]
    sk = [int(y_rare[te].sum()) for _, te in
          StratifiedKFold(5, shuffle=True, random_state=state).split(X_rare, y_rare)]
    flag = "   <-- a fold with NO positives" if 0 in kf else ""
    print(f"random_state={state}:  KFold {kf}   Stratified {sk}{flag}")

**4a.** I used `make_classification` with `weights=[0.94, 0.06]`, which gives 18 positives out of 300.

**4b and 4c.** `StratifiedKFold` gives the same balanced counts every time (3, 3, 4, 4, 4), whatever the random state. `KFold` wanders all over the place.

**4d.** With `random_state=1`, `KFold` made a fold with **zero** positives. Whether you see one depends on the seed, so you might not have, and that is fine. To make it more likely: make the positive class rarer, use more folds, or use a smaller dataset.

On a fold with zero positives, a model that answers "negative" every single time gets **100 percent accuracy**, because every case really is negative. That fold tells you nothing about whether the model can find the rare thing, and it pulls your average score up for a reason that has nothing to do with the model.

---

## Problem 5: A judgment call

No code. You are reviewing a colleague's work.

They built a model to predict which loan applicants will default. They report **94
percent accuracy** from 5-fold cross validation. Their pipeline scales the features
inside the pipeline, correctly. About **3 percent** of applicants in the data
actually defaulted.

**5a.** Before asking anything else, what is the one number you would want to compare
that 94 percent against? Compute it from what you have been told.

**5b.** Is 94 percent good? Defend your answer.

**5c.** They used `KFold`, not `StratifiedKFold`. Why does that matter more here than
it would on a balanced dataset?

**5d.** Name one further thing you would ask them for before you believed the model
was useful.

*There is more than one reasonable answer. I care about your reasoning.*

**5a.** The base rate. If 3 percent default, a model that always answers "will not default" is **97 percent accurate**. Their 94 percent is **worse than doing nothing**!

**5b.** No, 94 percent is bad. It is three points below a model that always gives the same answer. And it tells you nothing about whether the model finds the people who default, which is the whole reason the model exists.

**5c.** With only 3 percent positives, folds made without stratifying vary a lot in how many defaulters they get, and a fold could easily get very few or none. On a balanced dataset the folds all come out about the same by luck, so stratifying changes little. The rarer the class, the more stratification matters.

**5d.** Some good things to ask for: the spread across the folds, not just the mean; how many defaulters are in each fold; whether anything leaked, eg. a feature recorded after the default happened; and what a missed default costs compared to a wrongly refused loan. If you thought of precision and recall, you are a day early, that is exactly where Day 5 starts!